In [1]:
# ! pip install --upgrade ultralytics

In [2]:
from utils.multicamera_tools import parse_camera_xml, triangulate_poses
from utils.video_tools import get_camera_calibration_files, get_video_files
from scripts.frame_iterator import video_frame_iterator
from scripts.parsers import parse_sequences as parse_sequence_info
import numpy as np
import bvhio
import warnings
import json

warnings.filterwarnings('ignore')

with open("./datasets/yolo/selected_joint_names_v2.json", "r") as file:
    selected_joint_names = json.load(file)

file_path = 'gait3d\\ListOfSequences.txt'
sequences = parse_sequence_info(file_path)

In [3]:
from ultralytics import YOLO

model = YOLO("yolo26x-pose.pt")

In [8]:
FRAME_WIDTH = 960
FRAME_HEIGHT = 540

for i in range(1, 5):
    results = model.predict(
        source=f'./gait3d/Sequences/p5s1/Images/c{i}_0195.avi',
        show=False, # do not display during processing
        save=True, # save annotated video
        device=0, # to use gpu
        project='sample_vids',
        name='yolo26', 
        # exist_ok=True,
        verbose=False, 
        stream=True
    )

    for result in results:
        if not len(result.keypoints.xyn) == 1:
            xy_n == [[0, 0] for _ in range(17)]
        xy_n = result.keypoints.xyn[0].cpu().numpy()
        xy_abs = xy_n * [FRAME_WIDTH, FRAME_HEIGHT]
    
        # print(f"{xy_n = }")
        # print(f"{xy_abs }")
        # break

Results saved to C:\Users\Miko7\Studia\magisterka\gait-features-identification\runs\pose\sample_vids\yolo265
Results saved to C:\Users\Miko7\Studia\magisterka\gait-features-identification\runs\pose\sample_vids\yolo266
Results saved to C:\Users\Miko7\Studia\magisterka\gait-features-identification\runs\pose\sample_vids\yolo267
Results saved to C:\Users\Miko7\Studia\magisterka\gait-features-identification\runs\pose\sample_vids\yolo268


In [5]:
selected_joint_names = {int(key): value for key, value in selected_joint_names.items()}
selected_joint_names

{5: 'lhumerus',
 6: 'rhumerus',
 11: 'lfemur',
 12: 'rfemur',
 13: 'ltibia',
 14: 'rtibia',
 15: 'lfoot',
 16: 'rfoot',
 7: 'lradius',
 8: 'rradius',
 9: 'lwrist',
 10: 'rwrist'}

In [9]:
VIDEO_FPS = 25
MOCAP_FPS = 100
FRAME_TIME = 1000/VIDEO_FPS
FRAME_WIDTH = 960
FRAME_HEIGHT = 540
YOLO_LANDMARKS_NUM = 17

yolo_selection = {}
yolo_triangulation = {}

for seq_key in list(sequences.keys()):
    print(seq_key, end=' ')
    if sequences[seq_key]['MoCap_data']:
        video_files = get_video_files(seq_key)
        max_frames = sequences[seq_key]['number_of_frames']
                
        camera_files_paths = get_camera_calibration_files(seq_key)
        cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]
    
        predicted_for_seq = {f"c{i+1}": {} for i in range(4)}
        camera_landmarks_found = [[True, True, True, True] for j in range(max_frames)]
        
        combined_cameras_with_landmarks = []
        combined_triangulation_results = []
        
        for c_idx, c_file in enumerate(video_files):
            print(f"c{c_idx+1}", end=' ')
            results = model.predict(
                source=c_file,
                show=False, # do not display during processing
                save=False, # do not save annotated video
                project='sample_vids',
                name='yolo26', 
                verbose=False, 
                stream=True
            )

            
            for f_idx, result in enumerate(results):
                if len(result.keypoints.xyn) == 1 and len(result.keypoints.xyn[0] == YOLO_LANDMARKS_NUM):
                    xy_n = result.keypoints.xyn[0].cpu().numpy().tolist()
                    
                    for important_joint in selected_joint_names.keys():
                        if xy_n[important_joint] == [0, 0]:
                            xy_n == [[None, None] for _ in range(17)]
                            camera_landmarks_found[f_idx][c_idx] = False
                            # print(seq_key, c_idx, f_idx, selected_joint_names[important_joint])
                            break

                else:
                    xy_n == [[None, None] for _ in range(17)]
                    camera_landmarks_found[f_idx][c_idx] = False
                    # print(seq_key, c_idx, f_idx, len(result.keypoints.xyn))

                predicted_for_seq[f"c{c_idx+1}"][f_idx] = xy_n

        yolo_selection[seq_key] = predicted_for_seq

        for f_idx in range(max_frames):
            found_landmarks_cameras_idx = ([camera_i for camera_i, camera_l_found in 
                                            enumerate(camera_landmarks_found[f_idx])
                                            if camera_l_found])
            
            selected_cameras_params = [cameras_params[camera_i] for camera_i in found_landmarks_cameras_idx]
            found_2d_points = np.array([np.array(predicted_for_seq[f"c{camera_i+1}"][f_idx]) * [FRAME_WIDTH, FRAME_HEIGHT] for camera_i in found_landmarks_cameras_idx])
            triangulation_result = triangulate_poses(selected_cameras_params, found_2d_points)
            combined_triangulation_results.append(triangulation_result[0].tolist())

        yolo_triangulation[seq_key] = combined_triangulation_results

    print(' | ', end='')


p1s1 c1 c2 c3 c4  | p1s2 c1 c2 c3 c4  | p1s3 c1 c2 c3 c4  | p1s4 c1 c2 c3 c4  | p2s1 c1 c2 c3 c4  | p2s2 c1 c2 c3 c4  | p2s3 c1 c2 c3 c4  | p2s4 c1 c2 c3 c4  | p3s1 c1 c2 c3 c4  | p3s2 c1 c2 c3 c4  | p3s3 c1 c2 c3 c4  | p3s4 c1 c2 c3 c4  | p4s1 c1 c2 c3 c4  | p4s2 c1 c2 c3 c4  | p4s3 c1 c2 c3 c4  | p4s4 c1 c2 c3 c4  | p5s1 c1 c2 c3 c4  | p5s2 c1 c2 c3 c4  | p5s3 c1 c2 c3 c4  | p5s4 c1 c2 c3 c4  | p6s1 c1 c2 c3 c4  | p6s2 c1 c2 c3 c4  | p6s3 c1 c2 c3 c4  | p6s4 c1 c2 c3 c4  | p7s1 c1 c2 c3 c4  | p7s2 c1 c2 c3 c4  | p7s3 c1 c2 c3 c4  | p7s4 c1 c2 c3 c4  | p8s1 c1 c2 c3 c4  | p8s2 c1 c2 c3 c4  | p8s3 c1 c2 c3 c4  | p8s4 c1 c2 c3 c4  | p9s1 c1 c2 c3 c4  | p9s2 c1 c2 c3 c4  | p9s3 c1 c2 c3 c4  | p9s4 c1 c2 c3 c4  | p10s1 c1 c2 c3 c4  | p10s2 c1 c2 c3 c4  | p10s3 c1 c2 c3 c4  | p10s4 c1 c2 c3 c4  | p11s1 c1 c2 c3 c4  | p11s2 c1 c2 c3 c4  | p11s3 c1 c2 c3 c4  | p11s4 c1 c2 c3 c4  | p12s1 c1 c2 c3 c4  | p12s2 c1 c2 c3 c4  | p12s3 c1 c2 c3 c4  | p12s4 c1 c2 c3 c4  | p13s1 c1 c2 c3 c4  | p13s2 c

In [10]:
import json

with open("./datasets/yolo/dataset_yolo26.json", "w") as f:
    json.dump(yolo_selection, f, indent=4)

with open("./datasets/yolo/triangulation_yolo26.json", "w") as f:
    json.dump(yolo_triangulation, f, indent=4)

In [11]:
VIDEO_FPS = 25
MOCAP_FPS = 100
FRAME_TIME = 1000/VIDEO_FPS
FRAME_WIDTH = 960
FRAME_HEIGHT = 540
YOLO_LANDMARKS_NUM = 17

yolo_selection = {}
yolo_triangulation = {}

for seq_key in list(sequences.keys()):
    print(seq_key, end=' ')
    if sequences[seq_key]['MoCap_data']:
        video_files = get_video_files(seq_key)
        max_frames = sequences[seq_key]['number_of_frames']
                
        camera_files_paths = get_camera_calibration_files(seq_key)
        cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]
    
        predicted_for_seq = {f"c{i+1}": {} for i in range(4)}
        prediction_confidence_for_seq = {f"c{i+1}": {} for i in range(4)}
        combined_triangulation_results = []
        
        for c_idx, c_file in enumerate(video_files):
            print(f"c{c_idx+1}", end=' ')
            results = model.predict(
                source=c_file,
                show=False, # do not display during processing
                save=False, # do not save annotated video
                project='sample_vids',
                name='yolo26', 
                verbose=False, 
                stream=True
            )

            
            for f_idx, result in enumerate(results):
                if len(result.keypoints.xy) == 1 and len(result.keypoints.xy[0] == YOLO_LANDMARKS_NUM) and result.keypoints.conf is not None:
                    results_xy = result.keypoints.xy.cpu().numpy().tolist()
                    results_conf = result.keypoints.conf.cpu().numpy().tolist()
                    xy_n = results_xy[0]
                    xy_confidence = results_conf[0]

                else:
                    xy_n == [[None, None] for _ in range(YOLO_LANDMARKS_NUM)]
                    xy_confidence = [0 for _ in range(YOLO_LANDMARKS_NUM)]

                predicted_for_seq[f"c{c_idx+1}"][f_idx] = xy_n
                prediction_confidence_for_seq[f"c{c_idx+1}"][f_idx] = xy_confidence

        yolo_selection[seq_key] = predicted_for_seq

        for f_idx in range(max_frames):
            triangulation_result = []
            for landmark_idx in range(YOLO_LANDMARKS_NUM):
                min_conf_camera = prediction_confidence_for_seq["c1"][f_idx][landmark_idx]
                min_conf_camera_idx = 0
                for camera_idx in range(1,4):
                    if (curr_conf := prediction_confidence_for_seq[f"c{camera_idx+1}"][f_idx][landmark_idx]) < min_conf_camera:
                        min_conf_camera = curr_conf
                        min_conf_camera_idx = camera_idx

                selected_cameras_idx = [i for i in range(4) if i != min_conf_camera_idx]
                assert len(selected_cameras_idx) == 3
                selected_cameras_params = [cameras_params[camera_i] for camera_i in selected_cameras_idx]
                found_2d_points = np.array([[predicted_for_seq[f"c{camera_idx+1}"][f_idx][landmark_idx]] for camera_idx in selected_cameras_idx])
                landmark_triangulation_result = triangulate_poses(selected_cameras_params, found_2d_points)
                triangulation_result.append(landmark_triangulation_result[0][0].tolist())
            
            combined_triangulation_results.append(triangulation_result)

        yolo_triangulation[seq_key] = combined_triangulation_results
    print(' | ', end='')

p1s1 c1 c2 c3 c4  | p1s2 c1 c2 c3 c4  | p1s3 c1 c2 c3 c4  | p1s4 c1 c2 c3 c4  | p2s1 c1 c2 c3 c4  | p2s2 c1 c2 c3 c4  | p2s3 c1 c2 c3 c4  | p2s4 c1 c2 c3 c4  | p3s1 c1 c2 c3 c4  | p3s2 c1 c2 c3 c4  | p3s3 c1 c2 c3 c4  | p3s4 c1 c2 c3 c4  | p4s1 c1 c2 c3 c4  | p4s2 c1 c2 c3 c4  | p4s3 c1 c2 c3 c4  | p4s4 c1 c2 c3 c4  | p5s1 c1 c2 c3 c4  | p5s2 c1 c2 c3 c4  | p5s3 c1 c2 c3 c4  | p5s4 c1 c2 c3 c4  | p6s1 c1 c2 c3 c4  | p6s2 c1 c2 c3 c4  | p6s3 c1 c2 c3 c4  | p6s4 c1 c2 c3 c4  | p7s1 c1 c2 c3 c4  | p7s2 c1 c2 c3 c4  | p7s3 c1 c2 c3 c4  | p7s4 c1 c2 c3 c4  | p8s1 c1 c2 c3 c4  | p8s2 c1 c2 c3 c4  | p8s3 c1 c2 c3 c4  | p8s4 c1 c2 c3 c4  | p9s1 c1 c2 c3 c4  | p9s2 c1 c2 c3 c4  | p9s3 c1 c2 c3 c4  | p9s4 c1 c2 c3 c4  | p10s1 c1 c2 c3 c4  | p10s2 c1 c2 c3 c4  | p10s3 c1 c2 c3 c4  | p10s4 c1 c2 c3 c4  | p11s1 c1 c2 c3 c4  | p11s2 c1 c2 c3 c4  | p11s3 c1 c2 c3 c4  | p11s4 c1 c2 c3 c4  | p12s1 c1 c2 c3 c4  | p12s2 c1 c2 c3 c4  | p12s3 c1 c2 c3 c4  | p12s4 c1 c2 c3 c4  | p13s1 c1 c2 c3 c4  | p13s2 c

In [12]:
with open("./datasets/yolo/triangulation_best_cameras_yolo26.json", "w") as f:
    json.dump(yolo_triangulation, f, indent=4)

In [13]:
with open("./datasets/yolo/triangulation_best_cameras_yolo26.json", "r") as file:
    triangulation_data = json.load(file)

with open("./datasets/yolo/selected_joint_names_v2.json", "r") as file:
    selected_names = json.load(file)

scale_factor = 255
dataset_v3 = {key: [] for key in triangulation_data.keys()}
for key, sequence_data in triangulation_data.items():
    for frame in sequence_data:
        frame_parameters = {}
        for idx, joint_name in selected_names.items():
            frame_parameters[joint_name] = [num/scale_factor for num in frame[int(idx)]]
            
        dataset_v3[key].append(frame_parameters)

with open("./datasets/yolo/dataset_best_cameras_yolo26.json", "w") as f:
    json.dump(dataset_v3, f, indent=4)